In [1]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, Subset
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_sparse import SparseTensor

from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

/home/snu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def setup_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
data = np.load('/home/snu/Downloads/breastmnist_224.npz', allow_pickle=True)

images = np.concatenate(
    [data['train_images'], data['val_images'], data['test_images']], axis=0
).astype(np.float32) / 255.0

labels = np.concatenate(
    [data['train_labels'], data['val_labels'], data['test_labels']], axis=0
).squeeze().astype(np.int64)

# Convert to 3-channel
images = np.repeat(images[:, None, :, :], 3, axis=1)

X_img = torch.tensor(images)
y = labels

In [4]:
idx0 = np.where(y == 0)[0]
idx1 = np.where(y == 1)[0]

random.seed(42)
idx0 = random.sample(idx0.tolist(), min(1000, len(idx0)))
idx1 = random.sample(idx1.tolist(), min(1000, len(idx1)))

indices = idx0 + idx1
random.shuffle(indices)

X_img = X_img[indices]
y = y[indices]

dataset = TensorDataset(X_img, torch.tensor(y))
loader = DataLoader(dataset, batch_size=64, shuffle=False)

print("Balanced samples:", len(y))

Balanced samples: 780


In [5]:
vit = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')
vit.eval().to(device)

features = []
labels_list = []

with torch.no_grad():
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        feats = vit(imgs)
        features.append(feats.cpu())
        labels_list.extend(lbls.numpy())

X = torch.cat(features, dim=0).numpy().astype(np.float32)
y = np.array(labels_list)

N, F_dim = X.shape
print("Feature matrix:", X.shape)

x = torch.from_numpy(X).to(device)

Using cache found in /home/snu/.cache/torch/hub/facebookresearch_dino_main


Feature matrix: (780, 768)


In [6]:
def create_adj(features, alpha=0.7):
    f = features / np.linalg.norm(features, axis=1, keepdims=True)
    W = np.dot(f, f.T)
    W = (W >= alpha).astype(np.float32)
    return W

W = create_adj(X, alpha=0.7)

rows, cols = np.nonzero(W)
edge_index = torch.tensor([rows, cols], dtype=torch.long)
edge_index = to_undirected(edge_index).to(device)

adj = SparseTensor(
    row=edge_index[0],
    col=edge_index[1],
    sparse_sizes=(N, N)
).fill_value(1.).to(device)

print("Edges:", adj.nnz())

Edges: 351480


/tmp/ipykernel_1568492/4064267530.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:275.)
  edge_index = torch.tensor([rows, cols], dtype=torch.long)


In [7]:
def get_sim(batch, adj, wt=50, wl=2):
    batch_size = batch.shape[0]
    batch_repeat = batch.repeat(wt)
    rw = adj.random_walk(batch_repeat, wl)[:, 1:]
    rw = rw.t().reshape(-1, batch_size).t()

    row, col, val = [], [], []
    for i in range(batch_size):
        nodes, counts = torch.unique(rw[i], return_counts=True)
        row += [batch[i].item()] * nodes.shape[0]
        col += nodes.tolist()
        val += counts.tolist()

    adj_rw = SparseTensor(
        row=torch.tensor(row),
        col=torch.tensor(col),
        value=torch.tensor(val),
        sparse_sizes=(batch_size, batch_size)
    )
    return adj_rw.set_diag(0.)


In [8]:
def get_mask(adj):
    mean = adj.mean(dim=1)
    mask = (adj.storage.value() -
            mean[adj.storage.row()]) > -1e-10

    return SparseTensor(
        row=adj.storage.row()[mask],
        col=adj.storage.col()[mask],
        value=adj.storage.value()[mask],
        sparse_sizes=adj.sizes()
    )

def scale(z):
    zmin = z.min(dim=1, keepdim=True)[0]
    zmax = z.max(dim=1, keepdim=True)[0]
    return (z - zmin) / (zmax - zmin + 1e-12)

In [9]:
class MAGILoss(nn.Module):
    def __init__(self, tau=0.3):
        super().__init__()
        self.tau = tau

    def forward(self, z, mask):
        sim = torch.mm(z, z.t()) / self.tau
        sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

        logits_mask = torch.ones_like(sim) - torch.eye(z.size(0), device=z.device)
        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True))

        row, col = mask.storage.row(), mask.storage.col()
        return -log_prob[row, col].mean()

In [10]:
class Encoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()
        self.conv = GCNConv(in_dim, hidden_dim)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        return F.leaky_relu(x, 0.2)

In [11]:
from sklearn.metrics import normalized_mutual_info_score  # <-- added

n_runs  = 10
epochs  = 3000

acc_list, prec_list, rec_list, f1_list, nmi_list = [], [], [], [], []  # <-- added nmi_list

for run in range(n_runs):
    print(f"\n========== Run {run+1}/{n_runs} ==========")

    setup_seed(42 + run)

    encoder   = Encoder(F_dim, 256).to(device)
    criterion = MAGILoss(tau=0.3)
    optimizer = torch.optim.Adam(
        encoder.parameters(), lr=5e-4, weight_decay=1e-3
    )

    batch  = torch.arange(N, device=device)
    adj_rw = get_sim(batch, adj)
    mask   = get_mask(adj_rw)

    for epoch in range(epochs):
        encoder.train()
        optimizer.zero_grad()

        z    = encoder(x, edge_index)
        z    = scale(z)
        z    = F.normalize(z, dim=1)
        loss = criterion(z, mask)
        loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            print(f"Run {run+1} | Epoch {epoch} | Loss {loss.item():.4f}")

    # -------------------- Evaluation (final model only) --------------------
    encoder.eval()
    with torch.no_grad():
        z = encoder(x, edge_index)
        z = scale(z)
        z = F.normalize(z, dim=1).cpu().numpy()

    kmeans = KMeans(n_clusters=2, n_init=20, random_state=run)
    y_pred = kmeans.fit_predict(z)

    acc1 = accuracy_score(y, y_pred)
    acc2 = accuracy_score(y, 1 - y_pred)
    if acc2 > acc1:
        y_pred = 1 - y_pred

    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec  = recall_score(y, y_pred,    zero_division=0)
    f1   = f1_score(y, y_pred,        zero_division=0)

    # ── NMI: one value per run, final model only ──
    nmi  = normalized_mutual_info_score(y, y_pred, average_method='arithmetic')

    acc_list.append(acc)
    prec_list.append(prec)
    rec_list.append(rec)
    f1_list.append(f1)
    nmi_list.append(nmi)   # <-- added

    print(
        f"Run {run+1} → "
        f"ACC: {acc:.4f} | "
        f"PREC: {prec:.4f} | "
        f"REC: {rec:.4f} | "
        f"F1: {f1:.4f} | "
        f"NMI: {nmi:.4f}"    # <-- added
    )

print("\n===== MAGI + K-Means (10 Runs) =====")
print(f"ACC : {np.mean(acc_list):.4f} ± {np.std(acc_list):.4f}")
print(f"PREC: {np.mean(prec_list):.4f} ± {np.std(prec_list):.4f}")
print(f"REC : {np.mean(rec_list):.4f} ± {np.std(rec_list):.4f}")
print(f"F1  : {np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}")
print(f"NMI : {np.mean(nmi_list):.4f} ± {np.std(nmi_list):.4f}")   # <-- added


========== Run 1/10 ==========
Run 1 | Epoch 0 | Loss 6.6386
Run 1 | Epoch 500 | Loss 6.4935
Run 1 | Epoch 1000 | Loss 6.4851
Run 1 | Epoch 1500 | Loss 6.4846
Run 1 | Epoch 2000 | Loss 6.4837
Run 1 | Epoch 2500 | Loss 6.4828
Run 1 → ACC: 0.6628 | PREC: 0.7241 | REC: 0.8702 | F1: 0.7904 | NMI: 0.0018

========== Run 2/10 ==========
Run 2 | Epoch 0 | Loss 6.6375
Run 2 | Epoch 500 | Loss 6.4882
Run 2 | Epoch 1000 | Loss 6.4874
Run 2 | Epoch 1500 | Loss 6.4853
Run 2 | Epoch 2000 | Loss 6.4844
Run 2 | Epoch 2500 | Loss 6.4840
Run 2 → ACC: 0.6615 | PREC: 0.7237 | REC: 0.8684 | F1: 0.7895 | NMI: 0.0020

========== Run 3/10 ==========
Run 3 | Epoch 0 | Loss 6.6406
Run 3 | Epoch 500 | Loss 6.4862
Run 3 | Epoch 1000 | Loss 6.4829
Run 3 | Epoch 1500 | Loss 6.4830
Run 3 | Epoch 2000 | Loss 6.4828
Run 3 | Epoch 2500 | Loss 6.4816
Run 3 → ACC: 0.6615 | PREC: 0.7243 | REC: 0.8667 | F1: 0.7891 | NMI: 0.0016

========== Run 4/10 ==========
Run 4 | Epoch 0 | Loss 6.6412
Run 4 | Epoch 500 | Loss 6.4845
